# Lab 8: Hypothesis Testing — Diagnosing Underpowered Studies and P-Hacking
## ECON 5200: Causal Machine Learning & Applied Analytics
### Diagnosis-First Lab | 90 min Core + 20 min Extension

---

**Format:** This lab contains **deliberately flawed analyses**. Your job:
1. Run the code
2. Identify what is wrong (you are not told what to look for)
3. Fix the issue
4. Document your reasoning
5. Extend the corrected analysis

**Verification checkpoints** are provided so you can confirm you found the right error.

**Time:** Core (90 min) + Extension (20 min) + Challenge (optional, ungraded)

With the README and the GitHub submission, plan on about two hours, most of it after class.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI-assisted coding section: every line you add is yours, including the required "Write it yourself" function. The one AI use is the README prompt at the end, which asks for documentation, not code.

---

> ### What you write, and what you run
>
> **You write code:**
> - **Diagnosis 1, Steps A to D:** five blanks: the observed effect size, the pilot's power, the sample size it needed, and the size of the rerun.
> - **Diagnosis 2, Steps A to C:** three blanks: the BH correction, the chance of at least one false positive, and what each simulated replication records.
> - **Write it yourself (required, after Diagnosis 2):** a `bh_reject()` function and one line that uses it, about ten lines, in an empty cell. A check cell tells you when it is right. It is required.
> - **Extension, Methods 3 and 4:** two blanks, the statistic each permutation test compares.
> - **Challenge (optional, ungraded):** three blanks in the p-hacking simulation.
>
> **You write, in text cells:** a 2-3 sentence diagnosis after each Diagnosis, and two one-sentence answers in the Extension.
>
> **You run and read:** everything else. That is the analyst's and the researcher's analyses, the chart of the 20 p-values, the `power_utils` module, the heavy-tailed earnings data, Methods 1 and 2, and the camel plot. Read each cell before you run it: the diagnosis questions ask about what it prints.

In [ ]:
# Setup — run first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import permutation_test
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.multitest import multipletests

RANDOM_STATE = 42

## Diagnosis 1: The Underpowered Study (30 min)

A junior analyst runs a t-test on a small pilot study and concludes:
"There is no evidence that the intervention works."

**Something is wrong with this conclusion.** Run the code, diagnose the problem, fix it.

**Reading the prints.** In an f-string, `{x:.3f}` shows three decimals, `{x:.1f}`
one, and `{x:,.0f}` a whole number with thousands separators.
`'Reject H0' if p_value < 0.05 else 'Fail to reject H0'` picks one of the two
strings by the condition.

In [ ]:
# THE ANALYST'S STUDY — run as-is, then audit the conclusion

np.random.seed(RANDOM_STATE)

# 15 people per group. In the population, treatment adds $800 to earnings:
# control mean $5,000, treatment mean $5,800, both with SD $3,000
n_pilot = 15
control_pilot = np.random.normal(loc=5000, scale=3000, size=n_pilot)
treatment_pilot = np.random.normal(loc=5800, scale=3000, size=n_pilot)

# equal_var=False makes this Welch's t-test: the groups need not share one variance
t_stat, p_value = stats.ttest_ind(treatment_pilot, control_pilot, equal_var=False)
mean_diff = np.mean(treatment_pilot) - np.mean(control_pilot)

print("=== Analyst's Report ===")
print(f"Treatment mean: ${np.mean(treatment_pilot):,.0f}")
print(f"Control mean:   ${np.mean(control_pilot):,.0f}")
print(f"Difference:     ${mean_diff:,.0f}")
print(f"T-statistic:    {t_stat:.3f}")
print(f"P-value:        {p_value:.4f}")
print(f"\nDecision: {'Reject H0' if p_value < 0.05 else 'Fail to reject H0'}")
print()
print("ANALYST'S CONCLUSION: 'There is no evidence that the intervention")
print("works. We recommend discontinuing the program.'")

### YOUR DIAGNOSIS

1. **Is the null true, or is the test underpowered?** How can you tell?
2. **Compute the achieved power** of this study. What does it tell you?
3. **How many participants per group** would be needed to detect this effect with 80% power?
4. **Re-run the study** with the adequate sample size (simulate from the same population parameters) and report the result.

**Verification checkpoint:** The achieved power of the pilot study should be well below 0.50 (indicating severe underpowering). The required n per group to detect d~0.27 at 80% power should be in the range of 200-250.

### Steps A to D

`TTestIndPower().solve_power()` links four quantities: the effect size (Cohen's
$d$), the size of group 1 (`nobs1`), `alpha` and `power`. Give it any three and it
returns the fourth. `ratio` is the size of group 2 divided by the size of group 1.

In [ ]:
# YOUR TASK — fill in the blank
# Step A: the observed effect size

# We simulated the data, so the true effect size is known: $800 on an SD of $3,000
d_true = 800 / 3000

# Cohen's d: the difference in the pilot's means over their pooled standard deviation.
# With equal group sizes, the pooled variance is the average of the two ddof=1 variances.
d_observed = ____
print(f"Observed Cohen's d: {d_observed:.3f}")

In [ ]:
# YOUR TASK — fill in the blanks
# Step B: the achieved power of the pilot study

power_analysis = TTestIndPower()
achieved_power = power_analysis.solve_power(effect_size=____, nobs1=____, alpha=0.05, ratio=1.0)
print(f"Achieved power: {achieved_power:.3f} ({achieved_power*100:.1f}%)")

In [ ]:
# YOUR TASK — fill in the blank
# Step C: the n per group for 80% power, at the true effect size

n_required = ____
print(f"Required n per group: {n_required:.0f}")
print(f"The pilot had {n_pilot} per group — roughly {n_pilot/n_required*100:.0f}% of what was needed.")

In [ ]:
# YOUR TASK — fill in the blank
# Step D: rerun the study at that size, from the same population

np.random.seed(RANDOM_STATE)
n_adequate = ____        # people per group: n_required rounded up to a whole number (np.ceil, then int)
control_full = np.random.normal(loc=5000, scale=3000, size=n_adequate)
treatment_full = np.random.normal(loc=5800, scale=3000, size=n_adequate)
t_stat_full, p_value_full = stats.ttest_ind(treatment_full, control_full, equal_var=False)

print("=== Adequately Powered Study ===")
print(f"n per group: {n_adequate}")
print(f"T-statistic: {t_stat_full:.3f}")
print(f"P-value:     {p_value_full:.4f}")
print(f"Decision:    {'Reject H0 — effect detected!' if p_value_full < 0.05 else 'Fail to reject'}")

### Document Your Diagnosis

Write 2-3 sentences explaining what was wrong with the analyst's conclusion and what the correct interpretation should be.

*Your diagnosis here:*



---

## Diagnosis 2: The P-Hacked Result (30 min)

A researcher tests 20 demographic subgroups from a job training program and reports:
"We found a significant treatment effect in Subgroup 12."

**Something is wrong with this conclusion.** Run the code, diagnose the problem, fix it.

In [ ]:
# THE RESEARCHER'S ANALYSIS — run as-is, then audit the conclusion

np.random.seed(RANDOM_STATE)

# 20 subgroups, 40 people per arm. Every arm is drawn from the same N(5000, 3000),
# so the program has no effect in any subgroup.
n_subgroups = 20
n_per_arm = 40

subgroup_results = []
for sg in range(n_subgroups):
    treat_sg = np.random.normal(loc=5000, scale=3000, size=n_per_arm)
    ctrl_sg = np.random.normal(loc=5000, scale=3000, size=n_per_arm)
    t_s, p_s = stats.ttest_ind(treat_sg, ctrl_sg, equal_var=False)
    diff = np.mean(treat_sg) - np.mean(ctrl_sg)
    subgroup_results.append({"subgroup": sg + 1, "diff": diff, "t_stat": t_s, "p_value": p_s})

results_df = pd.DataFrame(subgroup_results)      # each dict becomes one row

# The researcher reports the subgroup with the smallest p-value:
# idxmin() gives the row label of the minimum, and .loc[label] gives that row
best = results_df.loc[results_df["p_value"].idxmin()]
sig_count = (results_df["p_value"] < 0.05).sum()   # .sum() of True/False counts the Trues

print("=== Researcher's Report ===")
print(f"Tested {n_subgroups} subgroups for treatment effects.")
print(f"\nKey finding: Subgroup {int(best['subgroup'])} shows a significant effect!")
print(f"  Difference: ${best['diff']:,.0f}")
print(f"  T-statistic: {best['t_stat']:.3f}")
print(f"  P-value: {best['p_value']:.4f}")
print(f"\nTotal significant subgroups: {sig_count} of {n_subgroups}")
print()
print("RESEARCHER'S CONCLUSION: 'We found a significant treatment effect")
print(f"in Subgroup {int(best['subgroup'])} (p={best['p_value']:.3f}). The intervention")
print("should be targeted to this population.'")

### YOUR DIAGNOSIS

1. **What is the multiple comparisons problem here?** Why is the researcher's conclusion wrong?
2. **Apply BH correction** to all 20 p-values. Does the "significant" finding survive?
3. **Compute the probability** of at least one false positive among 20 independent tests at alpha=0.05.
4. **Simulate:** If all 20 null hypotheses are true, what fraction of 1000 replications produce at least one p < 0.05?

**Verification checkpoint:** After BH correction, zero subgroups should remain significant. The theoretical probability of at least one false positive in 20 tests is ~64%. Your simulation should yield a fraction close to 0.64.

### Steps A to C

`multipletests(pvals, alpha=..., method=...)` corrects a whole set of p-values at
once. It returns four things: a True/False reject flag for each p-value, the
adjusted p-values, and two numbers this lab does not use. Its method names are
listed in `help(multipletests)`. In an unpacking such as `a, b, _, _ = ...`, the
`_` is the usual name for a value you do not need.

In [ ]:
# YOUR TASK — fill in the blank
# Step A: apply the Benjamini-Hochberg (BH) correction to the 20 p-values

raw_pvals = results_df["p_value"].values        # .values: the column as a NumPy array
reject_bh, pvals_bh, _, _ = multipletests(raw_pvals, alpha=0.05, method=____)
print(f"BH-corrected significant subgroups: {np.sum(reject_bh)}")

# Subgroups are numbered from 1 but arrays from 0, hence the - 1
print(f"The 'significant' finding in Subgroup {int(best['subgroup'])} survives? "
      f"{reject_bh[int(best['subgroup']) - 1]}")

In [ ]:
# YOUR TASK — fill in the blank
# Step B: the chance of at least one p < 0.05 among 20 independent tests of true nulls

p_at_least_one = ____
print(f"P(at least one false positive in {n_subgroups} tests) = {p_at_least_one:.3f}")

In [ ]:
# YOUR TASK — fill in the blank
# Step C: rerun the researcher's 20 tests 1,000 times, with every null true

np.random.seed(RANDOM_STATE)
n_reps = 1000
any_significant = []          # one True/False per replication
for rep in range(n_reps):
    pvals = []
    for sg in range(n_subgroups):
        a = np.random.normal(0, 1, size=n_per_arm)
        b = np.random.normal(0, 1, size=n_per_arm)
        _, p = stats.ttest_ind(a, b, equal_var=False)
        pvals.append(p)
    any_significant.append(____)

# The mean of True/False values is the share of Trues
print(f"Simulation: {np.mean(any_significant)*100:.1f}% of replications had >= 1 false positive")
print(f"Theory:     {p_at_least_one*100:.1f}%")

In [ ]:
# GIVEN — run as-is
# The 20 raw p-values, against the uncorrected and the Bonferroni thresholds

colors = ["red" if p < 0.05 else "steelblue" for p in raw_pvals]    # one colour per bar
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(1, n_subgroups + 1), raw_pvals, color=colors)
ax.axhline(y=0.05, color="red", linestyle="--", label="alpha = 0.05 (uncorrected)")
ax.axhline(y=0.05 / n_subgroups, color="orange", linestyle="--",
           label=f"Bonferroni threshold (0.05 / {n_subgroups} = {0.05 / n_subgroups:.4f})")
ax.set_xlabel("Subgroup")
ax.set_ylabel("P-value")
ax.set_title("Raw P-Values Across 20 Subgroups")
ax.set_xticks(range(1, n_subgroups + 1))
ax.legend()
plt.tight_layout()
plt.show()

### Document Your Diagnosis

Write 2-3 sentences explaining what was wrong with the researcher's conclusion and what the correct interpretation should be.

*Your diagnosis here:*



---

## Write it yourself (required — 25 min)

This is the one function in this lab you write from an empty cell, and it is
required. No AI: write it yourself.

Step A let statsmodels apply the Benjamini-Hochberg (BH) correction. The rule
itself is short. With $m$ p-values:

1. Sort them: $p_{(1)} \le p_{(2)} \le \dots \le p_{(m)}$.
2. Find the largest rank $k$ with $p_{(k)} \le \frac{k}{m}\,\alpha$.
3. Reject the hypotheses with the $k$ smallest p-values, and no others. If no
   rank passes, reject nothing.

In the empty cell below:
1. Write a function `bh_reject(pvals, alpha=0.05)` that returns one True/False
   per p-value, in the order the p-values came in. True means rejected.
2. Use it to set `planted_flags`: BH's decision on a version of Diagnosis 2 in
   which two subgroups really did respond. Take the first 18 p-values in
   `raw_pvals` and append 0.0005 and 0.004 (`np.append(a, b)` joins two arrays).

It takes about ten lines. `np.argsort(p)` gives the positions that would sort
`p`, which is how you get back to the original order. Do Diagnosis 2 Step A
first: the check compares your function with its `reject_bh`, then tries it on
a few small sets of p-values, where each common slip gives a different answer.
Until you write the cell, the check says "Not written yet"; once you do, it
fails with a message until your function is right.

In [ ]:
# WRITE IT YOURSELF — required: bh_reject and planted_flags

In [ ]:
# CHECK — for bh_reject above
# globals() holds every name defined so far in this notebook
if "bh_reject" not in globals():
    print("Not written yet: define bh_reject in the cell above, then run this check again.")
else:
    assert "reject_bh" in globals(), (
        "Do Diagnosis 2 Step A first: this check compares bh_reject with its reject_bh.")

    def flags(pvals, alpha=0.05):
        """bh_reject's answer at this alpha, as an array of True/False."""
        return np.asarray(bh_reject(np.array(pvals), alpha=alpha)).astype(bool)

    # try runs the block; if it raises an error, the except block runs instead
    try:
        flags([0.001, 0.9])
    except TypeError as err:
        raise AssertionError(
            f"bh_reject(pvals, alpha=0.05) raised TypeError: {err}. bh_reject should take "
            "alpha as an argument, as the task asks.") from None
    try:
        lab_flags = flags(raw_pvals)
    except Exception as err:
        raise AssertionError(
            f"bh_reject(raw_pvals) raised {type(err).__name__}: {err}. No p-value in "
            "Diagnosis 2 passes its threshold, so bh_reject should return all False.") from None
    assert lab_flags.shape == raw_pvals.shape, (
        f"bh_reject should return one True/False per p-value, 20 here; it returned {lab_flags.size}.")
    assert (lab_flags == reject_bh).all(), (
        f"On Diagnosis 2's p-values Step A's BH rejects {reject_bh.sum()}; "
        f"bh_reject rejects {lab_flags.sum()}.")

    got = flags([0.001, 0.2, 0.5, 0.9])
    assert got.tolist() == [True, False, False, False], (
        f"For [0.001, 0.2, 0.5, 0.9] bh_reject gives {got.tolist()}. 0.001 has rank 1, so its "
        "threshold is 1/4 x 0.05 = 0.0125 and BH rejects it. Do your ranks start at 1?")
    got = flags([0.9, 0.001, 0.5, 0.3])
    assert got.tolist() == [False, True, False, False], (
        f"For [0.9, 0.001, 0.5, 0.3] bh_reject gives {got.tolist()}; BH rejects only the "
        "second, 0.001. Return the flags in the order the p-values came in, not sorted.")
    got = flags([0.001, 0.02, 0.5, 0.9])
    assert got.tolist() == [True, True, False, False], (
        f"For [0.001, 0.02, 0.5, 0.9] bh_reject gives {got.tolist()}. 0.02 has rank 2, so its "
        "threshold is 2/4 x 0.05 = 0.025. Comparing every p-value with alpha/m is Bonferroni.")
    got = flags([0.001, 0.03, 0.031, 0.9])
    assert got.tolist() == [True, True, True, False], (
        f"For [0.001, 0.03, 0.031, 0.9] bh_reject gives {got.tolist()}. 0.03 misses its own "
        "threshold (0.025), but 0.031 passes its (0.0375), so the largest passing rank is 3 "
        "and BH rejects all three smallest.")
    got = flags([0.001, 0.04, 0.5, 0.9], alpha=0.10)
    assert got.tolist() == [True, True, False, False], (
        f"At alpha = 0.10, [0.001, 0.04, 0.5, 0.9] should give [True, True, False, False]; "
        f"bh_reject gives {got.tolist()}. Does alpha reach the thresholds?")

    assert "planted_flags" in globals(), (
        "bh_reject works. Now set planted_flags, as the task asks.")
    planted = np.append(raw_pvals[:18], [0.0005, 0.004])
    assert np.asarray(planted_flags).tolist() == flags(planted).tolist(), (
        "planted_flags is not bh_reject's answer on the planted p-values: the first 18 "
        "of raw_pvals, then 0.0005 and 0.004.")
    print(f"bh_reject matches Step A on Diagnosis 2: {lab_flags.sum()} of 20 rejected.")
    print(f"With two real effects planted: raw p < 0.05 in {np.sum(planted < 0.05)}, "
          f"BH rejects {np.sum(planted_flags)}, Bonferroni (p < 0.05/20) rejects "
          f"{np.sum(planted < 0.05 / 20)}.")

---

## Module Output: Save Power Analysis Utility (5 min)

Here is the finished module. It wraps Diagnosis 1's two `solve_power` calls, the
power at a given $n$ and the $n$ for a target power, in one function, and adds a
plot of power curves. Read it against Steps B and C, then run the three cells
below. They save it as `src/power_utils.py`, which later labs can import.

In [ ]:
# GIVEN — run as-is: make the src folder the next cell writes into
import os
os.makedirs("src", exist_ok=True)      # exist_ok: no error if it is already there

In [ ]:
%%writefile src/power_utils.py
"""Power analysis helpers for two-group t-tests (ECON 5200, Lab 8)."""

import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.power import TTestIndPower


def compute_power_summary(effect_size, n_obs, alpha=0.05, target_power=0.80, ratio=1.0):
    """Power at n_obs per group, and the n per group that reaches target_power.

    float() and bool() turn NumPy values into plain Python ones, which print plainly.
    """
    assert effect_size > 0, "Effect size must be positive"

    pa = TTestIndPower()
    achieved = pa.solve_power(effect_size=effect_size, nobs1=n_obs, alpha=alpha,
                              ratio=ratio, alternative="two-sided")
    n_required = pa.solve_power(effect_size=effect_size, alpha=alpha, power=target_power,
                                ratio=ratio, alternative="two-sided")

    return {
        "effect_size": effect_size,
        "n_obs": n_obs,
        "alpha": alpha,
        "achieved_power": float(round(achieved, 4)),
        "target_power": target_power,
        "n_required": int(np.ceil(n_required)),      # round up: whole people
        "adequately_powered": bool(achieved >= target_power),
    }


def plot_power_curves(effect_sizes=(0.2, 0.5, 0.8), alpha=0.05, max_n=500,
                      current_n=None, save_path=None):
    """Power against n per group, one line per effect size. Returns the figure."""
    pa = TTestIndPower()
    fig, ax = plt.subplots(figsize=(10, 6))
    sample_sizes = np.arange(10, max_n)

    for d in effect_sizes:
        powers = [pa.solve_power(effect_size=d, nobs1=n, alpha=alpha, ratio=1.0,
                                 alternative="two-sided") for n in sample_sizes]
        ax.plot(sample_sizes, powers, label=f"d={d}")

    ax.axhline(y=0.80, color="gray", linestyle="--", label="80% power")
    if current_n is not None:
        ax.axvline(x=current_n, color="red", linestyle=":", label=f"Current n={current_n}")

    ax.set_xlabel("Sample Size per Group")
    ax.set_ylabel("Statistical Power")
    ax.set_title("Power Curves by Effect Size")
    ax.legend(loc="lower right")
    ax.set_ylim(0, 1.05)
    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    return fig

In [ ]:
# GIVEN — run as-is: import the module and try it
import sys
import importlib

sys.path.insert(0, "src")              # let Python find modules in src/
import power_utils
importlib.reload(power_utils)          # re-read the file, in case you ran the cell above again

print(power_utils.compute_power_summary(effect_size=0.3, n_obs=50))

---

## Extension: Robust Alternatives (20 min)

When the normality assumption fails — as it routinely does for earnings data —
the standard t-test may produce unreliable p-values. Compare three approaches
on heavy-tailed simulated data.

In [ ]:
# GIVEN — run as-is

np.random.seed(RANDOM_STATE)
n_heavy = 80

# exp() of a normal draw is log-normal: right-skewed, like earnings.
# Treatment shifts log earnings by 0.3.
control_heavy = np.exp(np.random.normal(loc=8.5, scale=0.8, size=n_heavy))
treatment_heavy = np.exp(np.random.normal(loc=8.8, scale=0.8, size=n_heavy))

# One very high earner in each group makes the tails heavier still
control_heavy[0] = 150000
treatment_heavy[0] = 200000

# alpha here is the bars' transparency, so the overlap shows
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(control_heavy, bins=30, alpha=0.6, color="sienna", label="Control")
ax.hist(treatment_heavy, bins=30, alpha=0.6, color="steelblue", label="Treatment")
ax.set_xlabel("Earnings ($)")
ax.set_ylabel("Frequency")
ax.set_title("Heavy-Tailed Earnings Data (Log-Normal + Outliers)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Control:   mean=${np.mean(control_heavy):,.0f}, median=${np.median(control_heavy):,.0f}")
print(f"Treatment: mean=${np.mean(treatment_heavy):,.0f}, median=${np.median(treatment_heavy):,.0f}")

In each group the mean sits far above the median: the outliers and the long
right tail pull the mean up.

### Four tests on the same data

Methods 1 and 2 are t-tests. Welch's compares the means; Yuen's compares 20%
trimmed means, the means after dropping the lowest and highest 20% of each
group, and `stats.ttest_ind` does it through its `trim` argument.

Methods 3 and 4 are permutation tests. `permutation_test` shuffles the group
labels 9,999 times, recomputes your statistic on each shuffle, and reports how
often a shuffled statistic is as extreme as the real one. It hands your
statistic function many shuffles at once, so the function gets an `axis`
argument that it must pass on to every mean it takes. `stats.trim_mean` computes
a trimmed mean, and it takes `axis` too. If you see "zero-dimensional arrays
cannot be concatenated" or "operands could not be broadcast together", or
Method 3 or 4 gives a p-value below 0.001, one of your means is missing
`axis=axis`.

In [ ]:
# GIVEN — run as-is
# Methods 1 and 2: Welch's t-test and Yuen's trimmed t-test

print("=== Method 1: Standard Welch's T-Test ===")
t_welch, p_welch = stats.ttest_ind(treatment_heavy, control_heavy, equal_var=False)
print(f"T-statistic: {t_welch:.3f}, P-value: {p_welch:.4f}")

print("\n=== Method 2: Yuen's Trimmed T-Test (20% trim) ===")
t_yuen, p_yuen = stats.ttest_ind(treatment_heavy, control_heavy, equal_var=False, trim=0.2)
print(f"T-statistic: {t_yuen:.3f}, P-value: {p_yuen:.4f}")

In [ ]:
# YOUR TASK — fill in the blanks
# Methods 3 and 4: permutation tests of the two statistics


def stat_means(x, y, axis):
    """Difference in means, x minus y."""
    return ____


def stat_trimmed(x, y, axis):
    """Difference in 20% trimmed means, x minus y."""
    return ____


print("=== Method 3: Permutation Test on the DIFFERENCE IN MEANS ===")
res_mean = permutation_test((treatment_heavy, control_heavy), stat_means,
                            n_resamples=9999, alternative='two-sided', random_state=RANDOM_STATE)
print(f"Observed diff: ${res_mean.statistic:,.0f}, P-value: {res_mean.pvalue:.4f}")

print("\n=== Method 4: Permutation Test on the 20% TRIMMED MEAN ===")
res_trim = permutation_test((treatment_heavy, control_heavy), stat_trimmed,
                            n_resamples=9999, alternative='two-sided', random_state=RANDOM_STATE)
print(f"Observed diff: ${res_trim.statistic:,.0f}, P-value: {res_trim.pvalue:.4f}")

**Answer in one sentence each:**

1. Which two of the four agree, and what do those two have in common?
2. Which two disagree with them, and what do *those* two have in common? It is
   not the parametric/non-parametric split.

*Your answers here:*

1.
2.

**Verification checkpoint.** The measured p-values on this data (`seed=42`,
`n_heavy=80`, both outliers in place) are:

| Method | statistic | p-value |
|---|---|---:|
| 1. Welch's t-test | difference in **means** | **0.2650** |
| 2. Yuen's trimmed t-test (20%) | difference in **trimmed means** | **0.0066** |
| 3. Permutation test | difference in **means** | **0.3496** |
| 4. Permutation test | difference in **trimmed means** | **0.0056** |

If your four p-values match these, answer the two questions above.

---

## Challenge (Take-Home): Reproduce Brodeur's Camel

Brodeur et al. (2020) found a suspicious "two-humped camel" shape in the distribution
of z-statistics published in economics journals: a dip just below z=1.96 (the 5% threshold)
and a spike just above it. This is the statistical fingerprint of p-hacking.

Simulate this phenomenon. The challenge is optional and ungraded.

In [ ]:
# YOUR TASK — optional, ungraded: fill in the blanks
# Steps 1 and 2: simulate 5,000 studies' z-statistics, then p-hack them

np.random.seed(RANDOM_STATE)
n_studies = 5000

# Step 1: 70% of studies test a true null, z ~ N(0, 1); 30% a real effect, z ~ N(2.5, 1)
n_null = int(0.70 * n_studies)
n_real = n_studies - n_null
z_null = ____
z_real = ____
z_clean = np.concatenate([z_null, z_real])     # one array: the nulls, then the real effects

# Step 2: a study with z between 1.5 and 1.96 tries specifications until it
# crosses the line, which we model as a random push of 0.3 to 0.8
z_hacked = z_clean.copy()                      # a copy, so z_clean stays as it was
# True for each z between 1.5 and 1.96; & joins two conditions, each in parentheses
borderline_mask = ____
nudge = np.random.uniform(0.3, 0.8, size=borderline_mask.sum())    # one push per borderline study
z_hacked[borderline_mask] += nudge             # adds the pushes to the borderline studies only

In [ ]:
# GIVEN — run as-is
# The two distributions, and the share of "significant" results in each


def z_histogram(ax, z, title, color):
    """Histogram of z-statistics with the 5% and 10% two-sided cutoffs marked."""
    ax.hist(z, bins=np.arange(-4, 6, 0.15), color=color, edgecolor="white")
    ax.axvline(x=1.96, color="red", linestyle="--", label="z=1.96 (p=0.05)")
    ax.axvline(x=1.645, color="orange", linestyle="--", label="z=1.645 (p=0.10)")
    ax.set_xlabel("Z-statistic")
    ax.set_ylabel("Frequency")
    ax.set_title(title)
    ax.legend()


fig, axes = plt.subplots(1, 2, figsize=(14, 5))
z_histogram(axes[0], z_clean, "Clean Distribution (No P-Hacking)", "steelblue")
z_histogram(axes[1], z_hacked, "P-Hacked Distribution (Brodeur's Camel)", "sienna")
plt.tight_layout()
plt.show()

# |z| > 1.96 is significant at 5%, two-sided; the mean of True/False is the share of Trues
clean_sig = np.mean(np.abs(z_clean) > 1.96)
hacked_sig = np.mean(np.abs(z_hacked) > 1.96)
print(f"Fraction 'significant' (clean):  {clean_sig:.3f}")
print(f"Fraction 'significant' (hacked): {hacked_sig:.3f}")
print(f"Inflation: {(hacked_sig - clean_sig) / clean_sig * 100:.1f}% more 'significant' results")

---
## Digital Portfolio: your README

### Your README prompt
Copy and paste the prompt below into Claude or ChatGPT. **Do NOT ask the AI to write Python code — only documentation.** Replace every `[YOUR VALUE]` with the number from your own output before you paste it.

```
I need help writing a project description for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Diagnosed an underpowered pilot study: with 15 people per group it had
  [YOUR VALUE]% power against the true effect, and 80% power needed
  [YOUR VALUE] people per group. At that size the test gave p = [YOUR VALUE]
* Diagnosed a p-hacked subgroup result: after a Benjamini-Hochberg
  correction, [YOUR VALUE] of the 20 subgroups stayed significant
* Simulated 20 tests of true nulls: [YOUR VALUE]% of 1,000 replications had
  at least one p < 0.05, against [YOUR VALUE]% in theory
* Wrote my own Benjamini-Hochberg function and checked it against
  statsmodels; with two real effects planted, it flagged [YOUR VALUE]
  subgroups
* Compared four tests on heavy-tailed earnings data and found that they
  split by the statistic tested, not by parametric versus permutation
* (If I did the challenge) Simulated Brodeur's "camel": p-hacking raised the
  share of significant results from [YOUR VALUE] to [YOUR VALUE]

**Please write a README.md entry including:**
1. Project Title: Diagnosing Underpowered Studies and P-Hacking
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'comprehensive' or
'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ5200-lab08-hypothesis-testing`

**First time?** Read the **GitHub Setup and Repository Guide** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ5200-lab08-hypothesis-testing`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ5200-lab08-hypothesis-testing`, branch `main`,
   commit message `Lab 08 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the README prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. The `src/power_utils.py` module this lab saves is **not** pushed by step 2 —
   that step sends the notebook only. On the repo page use **Add file →
   Upload files**, drag the `.py` in, and **Commit changes**. It is part of
   what is graded.
5. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.